In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import folium
from folium import plugins
import xgboost as xgb
import lightgbm as lgb


In [2]:
def load_ecotourism_data(filepath='/content/ecotourism_dataset.csv'):
    df = pd.read_csv(filepath)
    np.random.seed(42)
    n_samples = 5000
    df = df.sample(n=n_samples, random_state=42)
    return df
    df.head()

In [3]:
n_samples = 5000
data = {
        'Site_ID': range(1, n_samples + 1),
        'Site_Name': [f'EcoSite_{i}' for i in range(1, n_samples + 1)],
        'Latitude': np.random.uniform(10, 50, n_samples),
        'Longitude': np.random.uniform(-130, -60, n_samples),
        'Country': np.random.choice(['USA', 'Canada', 'Mexico', 'Costa Rica', 'Brazil'], n_samples),
        'Biodiversity_Index': np.random.uniform(0.2, 1.0, n_samples),
        'Air_Quality_Index': np.random.uniform(20, 200, n_samples),
        'Elevation_m': np.random.uniform(0, 4000, n_samples),
        'Vegetation_Type': np.random.choice(['Tropical_Forest', 'Temperate_Forest', 'Grassland',
                                           'Desert', 'Wetland', 'Mountain'], n_samples),
        'Flood_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Drought_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Temperature_C': np.random.uniform(5, 40, n_samples),
        'Annual_Rainfall_mm': np.random.uniform(200, 3000, n_samples),
        'Soil_Type': np.random.choice(['Clay', 'Sand', 'Loam', 'Rocky', 'Volcanic'], n_samples),
        'Soil_Erosion_Risk': np.random.uniform(0, 1, n_samples),
        'Tourist_Capacity_Limit': np.random.uniform(100, 5000, n_samples),
        'Current_Tourist_Count': np.random.uniform(50, 4000, n_samples),
        'Accessibility_Score': np.random.uniform(0.1, 1.0, n_samples),
        'Human_Activity_Index': np.random.uniform(0.05, 0.95, n_samples),
        'Protected_Area_Status': np.random.choice([True, False], n_samples),
        'Conservation_Investment_USD': np.random.uniform(10000, 500000, n_samples),
        'Climate_Risk_Score': np.random.uniform(0.1, 0.9, n_samples)
    }
df = pd.DataFrame(data)
df.to_csv('eco_sites.csv', index=False)

In [4]:
n_samples = 5000
data = {
        'Site_ID': range(1, n_samples + 1),
        'Site_Name': [f'EcoSite_{i}' for i in range(1, n_samples + 1)],
        'Latitude': np.random.uniform(10, 50, n_samples),
        'Longitude': np.random.uniform(-130, -60, n_samples),
        'Country': np.random.choice(['USA', 'Canada', 'Mexico', 'Costa Rica', 'Brazil'], n_samples),
        'Biodiversity_Index': np.random.uniform(0.2, 1.0, n_samples),
        'Air_Quality_Index': np.random.uniform(20, 200, n_samples),
        'Elevation_m': np.random.uniform(0, 4000, n_samples),
        'Vegetation_Type': np.random.choice(['Tropical_Forest', 'Temperate_Forest', 'Grassland',
                                           'Desert', 'Wetland', 'Mountain'], n_samples),
        'Flood_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Drought_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Temperature_C': np.random.uniform(5, 40, n_samples),
        'Annual_Rainfall_mm': np.random.uniform(200, 3000, n_samples),
        'Soil_Type': np.random.choice(['Clay', 'Sand', 'Loam', 'Rocky', 'Volcanic'], n_samples),
        'Soil_Erosion_Risk': np.random.uniform(0, 1, n_samples),
        'Tourist_Capacity_Limit': np.random.uniform(100, 5000, n_samples),
        'Current_Tourist_Count': np.random.uniform(50, 4000, n_samples),
        'Accessibility_Score': np.random.uniform(0.1, 1.0, n_samples),
        'Human_Activity_Index': np.random.uniform(0.05, 0.95, n_samples),
        'Protected_Area_Status': np.random.choice([True, False], n_samples),
        'Conservation_Investment_USD': np.random.uniform(10000, 500000, n_samples),
        'Climate_Risk_Score': np.random.uniform(0.1, 0.9, n_samples)
    }
df = pd.DataFrame(data)
df['Vulnerability_Score'] = (
        0.25 * (1 - df['Biodiversity_Index']) +
        0.15 * (df['Air_Quality_Index'] / 200) +
        0.15 * df['Flood_Risk_Index'] +
        0.15 * df['Drought_Risk_Index'] +
        0.10 * df['Soil_Erosion_Risk'] +
        0.10 * (df['Temperature_C'] / 40) +
        0.10 * df['Climate_Risk_Score']
    )
df['Vulnerability_Score'] += np.random.normal(0, 0.05, n_samples)
df['Vulnerability_Score'] = np.clip(df['Vulnerability_Score'], 0, 1)
df['Risk_Category'] = pd.cut(
        df['Vulnerability_Score'],
        bins=[0, 0.33, 0.67, 1.0],
        labels=['Low', 'Medium', 'High']
    )
df.head()

,Site_ID,Site_Name,Latitude,Longitude,Country,Biodiversity_Index,Air_Quality_Index,Elevation_m,Vegetation_Type,Flood_Risk_Index,...,Soil_Erosion_Risk,Tourist_Capacity_Limit,Current_Tourist_Count,Accessibility_Score,Human_Activity_Index,Protected_Area_Status,Conservation_Investment_USD,Climate_Risk_Score,Vulnerability_Score,Risk_Category
0,1,EcoSite_1,15.139948,-108.644408,Mexico,0.910699,103.925811,2034.064199,Mountain,0.064677,...,0.513370,1426.823750,592.424289,0.215462,0.180183,False,471619.927854,0.114079,0.326693,Low
1,2,EcoSite_2,45.365321,-87.449594,Brazil,0.426258,174.372523,3963.018962,Grassland,0.933327,...,0.166574,764.186134,1291.241086,0.595381,0.398367,False,99664.213551,0.893543,0.514637,Medium
2,3,EcoSite_3,47.198727,-77.625840,Mexico,0.790480,159.713606,1486.255121,Temperate_Forest,0.665467,...,0.027973,4621.589421,1535.215453,0.585381,0.467136,True,487642.825614,0.851949,0.385272,Medium
3,4,EcoSite_4,10.641915,-97.228496,Brazil,0.262007,57.405153,1808.375853,Temperate_Forest,0.757448,...,0.722029,3919.686206,3204.099709,0.772236,0.808919,True,20538.712856,0.121555,0.656326,Medium
4,5,EcoSite_5,10.026946,-82.050633,Mexico,0.665738,83.505932,2109.090946,Desert,0.830251,...,0.457675,1661.343660,2341.984273,0.681805,0.330921,False,342717.948500,0.428345,0.501744,Medium


In [5]:
def create_comprehensive_eda(df):
 fig = plt.figure(figsize=(24, 20))

 # 1. Distribution of Vulnerability Score
 plt.subplot(4, 4, 1)
 plt.hist(df['Vulnerability_Score'], bins=40, alpha=0.7, color='skyblue', edgecolor='black')
 plt.title('Distribution of Vulnerability Scores', fontsize=12, fontweight='bold')
 plt.xlabel('Vulnerability Score')
 plt.ylabel('Frequency')

 # 2. Risk Category Distribution
 plt.subplot(4, 4, 2)
 risk_counts = df['Risk_Category'].value_counts()
 plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', startangle=90)
 plt.title('Risk Category Distribution', fontsize=12, fontweight='bold')

 # 3. Biodiversity vs Vulnerability
 plt.subplot(4, 4, 3)
 plt.scatter(df['Biodiversity_Index'], df['Vulnerability_Score'], alpha=0.6, c='green', s=2)
 plt.title('Biodiversity vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Biodiversity Index')
 plt.ylabel('Vulnerability Score')

 # 4. Air Quality Impact
 plt.subplot(4, 4, 4)
 plt.scatter(df['Air_Quality_Index'], df['Vulnerability_Score'], alpha=0.6, c='orange', s=2)
 plt.title('Air Quality vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Air Quality Index')
 plt.ylabel('Vulnerability Score')

 # 5. Temperature vs Vulnerability
 plt.subplot(4, 4, 5)
 plt.scatter(df['Temperature_C'], df['Vulnerability_Score'], alpha=0.6, c='red', s=2)
 plt.title('Temperature vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Temperature (°C)')
 plt.ylabel('Vulnerability Score')

 # 6. Elevation Distribution by Risk Category
 plt.subplot(4, 4, 6)
 for risk_cat in df['Risk_Category'].unique():
  if pd.notna(risk_cat):
   subset = df[df['Risk_Category'] == risk_cat]
   plt.hist(subset['Elevation_m'], alpha=0.6, label=risk_cat, bins=20)
 plt.title('Elevation by Risk Category', fontsize=12, fontweight='bold')
 plt.xlabel('Elevation (m)')
 plt.ylabel('Frequency')
 plt.legend()

 # 7. Flood Risk vs Drought Risk
 plt.subplot(4, 4, 7)
 scatter = plt.scatter(df['Flood_Risk_Index'], df['Drought_Risk_Index'],
 c=df['Vulnerability_Score'], cmap='Reds', alpha=0.6, s=3)
 plt.title('Flood vs Drought Risk', fontsize=12, fontweight='bold')
 plt.xlabel('Flood Risk Index')
 plt.ylabel('Drought Risk Index')
 plt.colorbar(scatter, shrink=0.8)

 # 8. Vegetation Type Analysis
 plt.subplot(4, 4, 8)
 veg_vuln = df.groupby('Vegetation_Type')['Vulnerability_Score'].mean().sort_values()
 plt.barh(veg_vuln.index, veg_vuln.values, color='lightgreen', edgecolor='black')
 plt.title('Avg Vulnerability by Vegetation', fontsize=12, fontweight='bold')
 plt.xlabel('Average Vulnerability Score')

 # 9. Geographic Distribution
 plt.subplot(4, 4, 9)
 scatter = plt.scatter(df['Longitude'], df['Latitude'],
 c=df['Vulnerability_Score'], cmap='RdYlBu_r', alpha=0.6, s=1)
 plt.title('Geographic Vulnerability Distribution', fontsize=12, fontweight='bold')
 plt.xlabel('Longitude')
 plt.ylabel('Latitude')
 plt.colorbar(scatter, shrink=0.8)

 # 10. Tourist Capacity vs Current Count
 plt.subplot(4, 4, 10)
 plt.scatter(df['Tourist_Capacity_Limit'], df['Current_Tourist_Count'],
 c=df['Vulnerability_Score'], cmap='viridis', alpha=0.6, s=2)
 plt.title('Tourist Capacity vs Current Count', fontsize=12, fontweight='bold')
 plt.xlabel('Capacity Limit')
 plt.ylabel('Current Count')

 # 11. Conservation Investment Impact
 plt.subplot(4, 4, 11)
 plt.scatter(df['Conservation_Investment_USD'], df['Vulnerability_Score'],
 alpha=0.6, c='purple', s=2)
 plt.title('Conservation Investment vs Risk', fontsize=12, fontweight='bold')
 plt.xlabel('Investment (USD)')
 plt.ylabel('Vulnerability Score')

 # 12. Protected Area Status
 plt.subplot(4, 4, 12)
 protected_vuln = df.groupby('Protected_Area_Status')['Vulnerability_Score'].mean()
 plt.bar(['Not Protected', 'Protected'], protected_vuln.values,
 color=['red', 'green'], alpha=0.7, edgecolor='black')
 plt.title('Protection Status vs Vulnerability', fontsize=12, fontweight='bold')
 plt.ylabel('Average Vulnerability Score')

 # 13. Correlation Heatmap
 plt.subplot(4, 4, 13)
 numeric_cols = df.select_dtypes(include=[np.number]).columns
 corr_matrix = df[numeric_cols].corr()

 # Select key correlations with vulnerability
 key_corr = corr_matrix['Vulnerability_Score'].sort_values(key=abs, ascending=False)[:10]
 plt.barh(range(len(key_corr)), key_corr.values, color='steelblue')
 plt.yticks(range(len(key_corr)), key_corr.index, fontsize=10)
 plt.title('Top Correlations with Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Correlation Coefficient')

 # 14. Soil Type Distribution
 plt.subplot(4, 4, 14)
 soil_counts = df['Soil_Type'].value_counts()
 plt.pie(soil_counts.values, labels=soil_counts.index, autopct='%1.1f%%')
 plt.title('Soil Type Distribution', fontsize=12, fontweight='bold')

 # 15. Human Activity vs Vulnerability
 plt.subplot(4, 4, 15)
 plt.scatter(df['Human_Activity_Index'], df['Vulnerability_Score'],
 alpha=0.6, c='brown', s=2)
 plt.title('Human Activity vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Human Activity Index')
 plt.ylabel('Vulnerability Score')

 # 16. Accessibility Score Distribution
 plt.subplot(4, 4, 16)
 plt.hist(df['Accessibility_Score'], bins=30, alpha=0.7, color='gold', edgecolor='black')
 plt.title('Accessibility Score Distribution', fontsize=12, fontweight='bold')
 plt.xlabel('Accessibility Score')
 plt.ylabel('Frequency')

 plt.tight_layout()
 plt.savefig('comprehensive_eda_analysis.png', dpi=300, bbox_inches='tight')
 plt.show()
 create_comprehensive_eda(df)

In [6]:
from sklearn.preprocessing import LabelEncoder

def preprocess_data(df, target_column='Vulnerability_Score', task_type='regression'):
    print("\n" + "="*50)
    print(f"DATA PREPROCESSING - {task_type.upper()}")
    print("="*50)
    processed_df = df.copy()

    # Handle missing values
    if processed_df.isnull().sum().sum() > 0:
        numeric_columns = processed_df.select_dtypes(include=[np.number]).columns
        categorical_columns = processed_df.select_dtypes(include=['object', 'category']).columns
        # Fill numeric columns with median
        for col in numeric_columns:
            if processed_df[col].isnull().sum() > 0:
                processed_df[col].fillna(processed_df[col].median(), inplace=True)
        # Fill categorical columns with mode
        for col in categorical_columns:
            if processed_df[col].isnull().sum() > 0:
                processed_df[col].fillna(processed_df[col].mode()[0], inplace=True)

    # Prepare feature matrix X
    X = processed_df.copy()

    # Encode categorical variables
    categorical_columns_to_encode = ['Vegetation_Type', 'Soil_Type', 'Country']
    label_encoders = {}
    for col in categorical_columns_to_encode:
        if col in X.columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            label_encoders[col] = le
            print(f" Encoded {col}: {len(le.classes_)} categories")

    # Handle boolean columns
    if 'Protected_Area_Status' in X.columns:
        X['Protected_Area_Status'] = X['Protected_Area_Status'].astype(int)

    # Remove non-feature columns
    columns_to_remove = ['Site_ID', 'Site_Name', target_column]
    if task_type == 'classification' and 'Risk_Category' in X.columns:
        columns_to_remove.append('Risk_Category')
    elif task_type == 'regression' and 'Risk_Category' in X.columns: # Exclude Risk_Category from features for regression
         columns_to_remove.append('Risk_Category')


    for col in columns_to_remove:
        if col in X.columns:
            X = X.drop(col, axis=1)

    # Prepare target variable y
    if task_type == 'regression':
        y = processed_df[target_column]
        print(f" Target variable: {target_column} (continuous)")
    else:  # classification
        if 'Risk_Category' in processed_df.columns:
            le_target = LabelEncoder()
            y = le_target.fit_transform(processed_df['Risk_Category'])
            print(f" Target variable: Risk_Category (categorical)")
            print(f"   Classes: {le_target.classes_}")
        else:
            # Create categories from continuous target
            y = pd.cut(processed_df[target_column], bins=3, labels=[0, 1, 2])
            print(f" Target variable: Created from {target_column} (3 categories)")

    print(f"\n📊 Preprocessing Summary:")
    print(f"   Features: {X.shape[1]}")
    print(f"   Samples: {X.shape[0]}")


    return X, y, label_encoders

# Preprocess data for both regression and classification
X_reg, y_reg, encoders_reg = preprocess_data(df, task_type='regression')
X_clf, y_clf, encoders_clf = preprocess_data(df, task_type='classification')


DATA PREPROCESSING - REGRESSION
 Encoded Vegetation_Type: 6 categories
 Encoded Soil_Type: 5 categories
 Encoded Country: 5 categories
 Target variable: Vulnerability_Score (continuous)

📊 Preprocessing Summary:
   Features: 20
   Samples: 5000

DATA PREPROCESSING - CLASSIFICATION
 Encoded Vegetation_Type: 6 categories
 Encoded Soil_Type: 5 categories
 Encoded Country: 5 categories
 Target variable: Risk_Category (categorical)
   Classes: ['High' 'Low' 'Medium']

📊 Preprocessing Summary:
   Features: 20
   Samples: 5000


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def split_and_scale_data(X, y, test_size=0.2, task_type='regression'):
    print("\n" + "="*50)
    print(f"DATA SPLITTING AND SCALING - {task_type.upper()}")
    print("="*50)

    # Train-test split
    if task_type == 'classification':
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

    # Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f" Data split completed:")
    print(f"   Training set: {X_train.shape}")
    print(f"   Test set: {X_test.shape}")
    print(f"   Feature scaling: StandardScaler applied")

    return X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler

# Split and scale data for both tasks
X_train_reg, X_test_reg, y_train_reg, y_test_reg, X_train_reg_scaled, X_test_reg_scaled, scaler_reg = split_and_scale_data(X_reg, y_reg, task_type='regression')
X_train_clf, X_test_clf, y_train_clf, y_test_clf, X_train_clf_scaled, X_test_clf_scaled, scaler_clf = split_and_scale_data(X_clf, y_clf, task_type='classification')


DATA SPLITTING AND SCALING - REGRESSION
 Data split completed:
   Training set: (4000, 20)
   Test set: (1000, 20)
   Feature scaling: StandardScaler applied

DATA SPLITTING AND SCALING - CLASSIFICATION
 Data split completed:
   Training set: (4000, 20)
   Test set: (1000, 20)
   Feature scaling: StandardScaler applied


In [17]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import xgboost as xgb


def train_regression_models(X_train, X_test, y_train, y_test):
   # Define models with hyperparameter grids
    models = {
        'Linear Regression': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', LinearRegression())
            ]),
            'params': {}
        },

        'XGBoost': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', xgb.XGBRegressor(random_state=42, eval_metric='rmse'))
            ]),
            'params': {
                'regressor__n_estimators': [100, 200],
                'regressor__max_depth': [3, 6],
                'regressor__learning_rate': [0.1, 0.01]
            }
        },

        'Support Vector Regression': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', SVR())
            ]),
            'params': {
                'regressor__C': [1, 10],
                'regressor__kernel': ['rbf', 'linear']
            }
        }
    }

    results = {}
    best_models = {}
    for name, model_config in models.items():
        print(f" Training {name}...")

        try:
            if model_config['params']:
                # Grid search for hyperparameter tuning
                grid_search = GridSearchCV(
                    model_config['model'],
                    model_config['params'],
                    cv=5,
                    scoring='r2',
                    n_jobs=-1
                )
                grid_search.fit(X_train, y_train)
                best_model = grid_search.best_estimator_
                print(f"   Best params: {grid_search.best_params_}")
            else:
                best_model = model_config['model']
                best_model.fit(X_train, y_train)

            # Make predictions
            train_pred = best_model.predict(X_train)
            test_pred = best_model.predict(X_test)

            # Calculate metrics
            train_r2 = r2_score(y_train, train_pred)
            test_r2 = r2_score(y_test, test_pred)
            rmse = np.sqrt(mean_squared_error(y_test, test_pred))
            mae = np.mean(np.abs(y_test - test_pred))

            # Cross-validation score
            cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')

            results[name] = {
                'train_r2': train_r2,
                'test_r2': test_r2,
                'rmse': rmse,
                'mae': mae,
                'cv_mean': cv_scores.mean(),
                'cv_std': cv_scores.std(),
                'predictions': test_pred
            }

            best_models[name] = best_model

            print(f" Train R²: {train_r2:.4f}")
            print(f" Test R²: {test_r2:.4f}")
            print(f" RMSE: {rmse:.4f}")
            print(f" MAE: {mae:.4f}")
            print(f" CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

        except Exception as e:
            print(f" Error training {name}: {str(e)}")
            continue

    return results, best_models
# Train regression models
regression_results, regression_models = train_regression_models(X_train_reg, X_test_reg, y_train_reg, y_test_reg)

 Training Linear Regression...
 Train R²: 0.8077
 Test R²: 0.8082
 RMSE: 0.0497
 MAE: 0.0397
 CV Score: 0.8051 (+/- 0.0118)
 Training XGBoost...
   Best params: {'regressor__learning_rate': 0.1, 'regressor__max_depth': 3, 'regressor__n_estimators': 200}
 Train R²: 0.8686
 Test R²: 0.7867
 RMSE: 0.0524
 MAE: 0.0419
 CV Score: 0.7826 (+/- 0.0156)
 Training Support Vector Regression...
   Best params: {'regressor__C': 10, 'regressor__kernel': 'linear'}
 Train R²: 0.8051
 Test R²: 0.8053
 RMSE: 0.0501
 MAE: 0.0398
 CV Score: 0.8006 (+/- 0.0122)


In [10]:
X = df.drop(['Site_ID', 'Site_Name', 'Country', 'Vegetation_Type', 'Soil_Type', 'Risk_Category', 'Vulnerability_Score'], axis=1)
y = df['Risk_Category']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

In [13]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')

Accuracy: 0.8690
Precision: 0.8616
Recall: 0.8690
F1-score: 0.8293
